In [ ]:
from __future__ import division, print_function
import os
import scipy
import modelrev
from scipy import misc
import glob
import nibabel as nb
from collections import defaultdict
import os, pickle, sys
import shutil
from functools import partial
from keras import backend as K
import skimage.morphology
from skimage.exposure import equalize_adapthist, equalize_hist
import tensorflow as tf
import numpy as np
from distutils.util import strtobool
from random import shuffle
from pathlib import Path
import cv2
import pickle
import os.path
import time
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import openpyxl

home_dir = ''
new_dir = ''

def retreive_filelist(extension, path, include_subfolders=True):
    """
    Returns a list with all the files of a certain type in path
    :param extension: what extension to search for
    :param include_subfolders: whether to include subfolders
    :param path: specified path. otherwise use data root
    :return:
    """
    if include_subfolders:
        extension = ('**/**/*.%s' % extension)

    # Otherwise just search this folder
    else:
        extension = ('*.%s' % extension)

    # Join the pathnames
    path = os.path.join(path, extension)

    # Return the list of filenames
    return glob.glob(path)

def window_image(volume, level, width):
    """
    Windows the image. For images that come from CT scans
    :param volume: ndarray, Source image
    :param level: the center of the window
    :param width: how wide the window is +-
    :return: the windowed image
    """

    # Set the min and max HU
    minimum = int(level - width)
    maximum = int(level + width)

    # Fit to max
    volume[volume > maximum] = maximum

    # fit to min
    volume[volume < minimum] = minimum

    # Return the windowed image
    return volume

def zoom_2d(image, new_shape):
    """
    Uses open CV to resize a 2D image
    :param image: The input image, numpy array
    :param new_shape: New shape, tuple or array
    :return: the resized image
    """
    return cv2.resize(image, (new_shape[0], new_shape[1]), interpolation=cv2.INTER_NEAREST)

def load_NIFTY(path, reshape=True):
    """
    This function loads a .nii.gz file into a numpy array with dimensions Z, Y, X, C
    :param filename: path to the file
    :param reshape: whether to reshape the axis from/to ZYX
    :return:
    """

    # try:

    # Load the data from the nifty file
    raw_data = nb.load(path)

    # Reshape the image data from NiB's XYZ to numpy's ZYXC
    if reshape:
        data = reshape_NHWC(raw_data.get_fdata(), False)
    else:
        data = raw_data.get_fdata()

    # Return the data
    return data

def smooth_images(imgs, t_step=0.125, n_iter=5):
    """
    Curvature driven image denoising.
    In my experience helps significantly with segmentation.
    """

    for mm in range(len(imgs)):
        img = sitk.GetImageFromArray(imgs[mm])
        img = sitk.CurvatureFlow(image1=img,
                                        timeStep=t_step,
                                        numberOfIterations=n_iter)

        imgs[mm] = sitk.GetArrayFromImage(img)


    return imgs

def volume_computation_function (y):
    num_vox = np.sum(y)
    x_depth = 1
    y_depth = 1
    z_depth = 1
    area = num_vox* x_depth* y_depth*z_depth
    volume = area /1000
    return volume

def resize_pred_to_val(y_pred, shape):
    row = shape[1]
    col =  shape[2]

    resized_pred = np.zeros(shape)
    for mm in range(len(y_pred)):
        resized_pred[mm,:,:] =  cv2.resize( y_pred[mm,:,:,0], (row, col), interpolation=cv2.INTER_NEAREST)

    return resized_pred.astype(int)


def reshape_NHWC(vol, NHWC):
    """
    Method to reshape 2D or 3D tensor into Tensorflow's NHWC format
    vol: The image data
    NHWC whether the input has a channel dimension
    """

    # If this is a 2D image
    if len(vol.shape) == 2:
        # Create an extra channel at the beginning
        vol = np.expand_dims(vol, axis=0)

    # If there are 3 dimensions to the shape (2D with channels or 3D)
    if len(vol.shape) == 3:

        # If there is no channel dimension (i.e. grayscale)
        if not NHWC:
            # Move the last axis (Z) to the first axis
            vol = np.moveaxis(vol, -1, 0)

        # Create another axis at the end for channel
        vol = np.expand_dims(vol, axis=3)

    return vol

def load_images():
    filenames1, filenames = retreive_filelist('nii.gz', home_dir , include_subfolders=False), []
    for z in range(len(filenames1)):
        filenames.append(filenames1[z])
    filenames.sort()
    #print(len(filenames), ' files found. Examples: ', filenames)

    # global variables
    images = []
    masks = []

    for file in filenames:
        #label is not relevant for your case
        #label = file.split('.')[0] + '_Label_1.nii.gz'
        try:
            volume = np.squeeze(load_NIFTY(file))
            #label_file = np.squeeze(load_NIFTY(label))
            print('loading:', file)
        except:
            print('Unable to load: ', file)
            continue
        volume = window_image(volume, 80, 80)
        #segments = label_file
        for z in range(volume.shape[0]):
            if ((z<85) or (z>184)): continue
            data_image = zoom_2d(volume[z].astype(np.float32),
                                        [256, 256])
            #data_label = zoom_2d(segments[z].astype(np.float32),
                                        #[256, 256])


            images.append(data_image)
            #masks.append(data_label)
    X_val = np.array(images)
    #y_val = np.array(masks)
    X_val = np.concatenate(X_val, axis=0).reshape(-1, 256, 256, 1)
    X_val = smooth_images(X_val)

    #y_val = np.concatenate(y_val, axis=0).reshape(-1, 256, 256, 1)
    print(X_val.shape)
    #print(y_val.shape)
    yield X_val

def segment():
    start_ind = 0
    end_ind = 0
    end_ind += 100
    study = 1
    data = {}
    index = 0
    for X_val in load_images():
        #if converting to python "model1 = Unet(...))"
        model1 = modelrev.UNet((256, 256,1), start_ch=1, depth=7, batchnorm=True, dropout=0.0, maxpool=True, residual=True)
        model1.summary()
        model1.load_weights('')
        y_pred = model1.predict(X_val, verbose=1, batch_size=1)
        for mm in range(len(X_val)//100):
            shape = [100, 256, 256]
            y_pred_up = y_pred[start_ind:end_ind, :, :]
            #y_val_up = y_val[start_ind:end_ind, :, :]
            volumepredict = volume_computation_function(y_pred_up)
            #volumelabel = volume_computation_function(y_val_up)
            #it would be helpful if you code in here a column for 'baseline_volume' and a column for '24_hr volume')
            data[index] = {'id': study, 'predicted_volume': volumepredict}
            pred_segment = nb.Nifti1Image(y_pred_up, affine=np.eye(4), dtype=np.float32)
            print(pred_segment.shape)
            output_file = (new_dir + 'M' + str(study) + '.nii.gz')
            nb.save(pred_segment, output_file)
            #edit output_file to correct path for your image
            start_ind = end_ind
            end_ind += 100
            study +=1
            index +=1
        del volumepredict


    df = pd.DataFrame(data)
    df = df.transpose()
    df.to_excel('', index=False)

segment()